# Phase 6 - Telecom Infrastructure Heat Risk Assessment

Uses your predicted LST rasters (Phase 3/4 outputs) to flag which telecom sites -- cell towers,
exchanges, cabinets -- face the highest thermal stress risk. Sustained high ambient temperature
increases equipment failure rates and cooling costs for outdoor telecom cabinets and base
stations, so this is a genuine infrastructure-resilience use of the heat model, not just an
add-on.

**Pipeline:**
1. Load your predicted LST raster(s) (baseline, and optionally a scenario delta)
2. Get telecom site locations for your AOI (OpenCelliD -- see note below on data limitations)
3. Sample predicted LST at each site's coordinates
4. Classify each site into a thermal risk tier against equipment operating thresholds
5. (Optional) Adjust risk under a development scenario using the Phase 4 delta raster
6. Export results as CSV/GeoJSON for QGIS, plus a quick summary map

**Data caveat, worth stating explicitly in any writeup:** there is no public official MCMC
tower-location dataset for Malaysia (reasonable, given it's sensitive infrastructure). This
notebook uses OpenCelliD (opencellid.org), a crowd-sourced, CC BY-SA licensed open database --
a reasonable proxy, but not exhaustive or official. Treat results as indicative of *where*
thermal risk concentrates, not as a complete inventory of every real site.

**Prerequisite:** run this only after re-running Phases 2-5 against your corrected Klang Valley
`AOI` -- the paths below assume `gis_ready/` rasters from that corrected run.

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

DRIVE_FOLDER = "UHI_Phase2_Exports"
export_dir = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
gis_ready_dir = f"{export_dir}/gis_ready"
telecom_dir = f"{export_dir}/telecom_risk"
os.makedirs(telecom_dir, exist_ok=True)

# Which scenario's baseline raster to assess against. "industrial_park" and
# "greening_corridor" both share the same baseline_lst.tif (same AOI, same model) --
# pick either folder for the baseline; only the delta differs.
BASELINE_LST_PATH = f"{gis_ready_dir}/industrial_park/baseline_lst.tif"
SCENARIO_DELTA_PATH = f"{gis_ready_dir}/industrial_park/delta_lst.tif"  # optional, for step 5

print("Baseline raster:", BASELINE_LST_PATH, "exists:", os.path.exists(BASELINE_LST_PATH))

## 1. Load the predicted LST raster
Confirms the raster's CRS -- your Phase 2 config used `TARGET_CRS = "EPSG:4326"`, so raster coordinates are already plain lat/lon degrees, matching typical tower-location data directly with no reprojection needed. If you changed `TARGET_CRS`, adjust the coordinate transform in step 3 accordingly.

In [ ]:
with rasterio.open(BASELINE_LST_PATH) as src:
    baseline_lst = src.read(1)
    baseline_transform = src.transform
    baseline_crs = src.crs
    baseline_bounds = src.bounds

print("Raster CRS:", baseline_crs)
print("Raster bounds (lon/lat if EPSG:4326):", baseline_bounds)
print("Raster shape:", baseline_lst.shape)
print("LST range: {:.1f} to {:.1f} C".format(np.nanmin(baseline_lst), np.nanmax(baseline_lst)))

## 2. Get telecom site locations (OpenCelliD)

**Option A -- API download (run in Colab):** sign up for a free API token at
[opencellid.org](https://opencellid.org/), then use it below to pull Malaysia (MCC 502) cell
records filtered to your AOI bounding box.

**Option B -- manual CSV:** download Malaysia's cell CSV from
[opencellid.org/downloads.php](https://www.opencellid.org/downloads.php) after registering, save
it to Drive, and point `MANUAL_CSV_PATH` at it instead.

Either way you end up with a DataFrame with `lon`, `lat` columns to sample the raster at.

In [ ]:
OPENCELLID_API_TOKEN = ""  # paste your free token here, or leave blank to use MANUAL_CSV_PATH
MANUAL_CSV_PATH = ""        # e.g. f"{export_dir}/telecom_risk/502.csv" if downloaded manually

MCC_MALAYSIA = 502

if OPENCELLID_API_TOKEN:
    import requests

    # OpenCelliD's cells-in-area endpoint; bbox order is (lonMin, latMin, lonMax, latMax)
    lonMin, latMin, lonMax, latMax = (
        baseline_bounds.left, baseline_bounds.bottom, baseline_bounds.right, baseline_bounds.top
    )
    url = "https://opencellid.org/cell/getInArea"
    params = {
        "key": OPENCELLID_API_TOKEN,
        "BBOX": f"{latMin},{lonMin},{latMax},{lonMax}",
        "format": "json",
        "limit": 10000,
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    cells_json = resp.json().get("cells", [])
    towers_df = pd.DataFrame(cells_json)
    towers_df = towers_df.rename(columns={"lon": "lon", "lat": "lat"})
    print(f"Downloaded {len(towers_df)} cell records from OpenCelliD API")

elif MANUAL_CSV_PATH and os.path.exists(MANUAL_CSV_PATH):
    # OpenCelliD's bulk CSV columns: radio,mcc,net,area,cell,unit,lon,lat,range,samples,...
    towers_df = pd.read_csv(MANUAL_CSV_PATH)
    towers_df = towers_df[towers_df["mcc"] == MCC_MALAYSIA]
    towers_df = towers_df[
        (towers_df["lon"] >= baseline_bounds.left) & (towers_df["lon"] <= baseline_bounds.right) &
        (towers_df["lat"] >= baseline_bounds.bottom) & (towers_df["lat"] <= baseline_bounds.top)
    ]
    print(f"Loaded {len(towers_df)} cell records within AOI from manual CSV")

else:
    raise ValueError(
        "Set OPENCELLID_API_TOKEN (free signup at opencellid.org) or MANUAL_CSV_PATH "
        "to a downloaded CSV before continuing."
    )

towers_df = towers_df.drop_duplicates(subset=["lon", "lat"]).reset_index(drop=True)
print(towers_df[["lon", "lat"]].describe())

## 3. Sample predicted LST at each site's coordinates
Direct pixel lookup via the raster's own transform -- valid here because `TARGET_CRS` is `EPSG:4326`, the same lon/lat degrees OpenCelliD reports in.

In [ ]:
def sample_raster_at_points(raster_array, transform, lons, lats):
    rows, cols = rasterio.transform.rowcol(transform, lons, lats)
    rows = np.asarray(rows)
    cols = np.asarray(cols)

    values = np.full(len(lons), np.nan, dtype=np.float32)
    in_bounds = (
        (rows >= 0) & (rows < raster_array.shape[0]) &
        (cols >= 0) & (cols < raster_array.shape[1])
    )
    values[in_bounds] = raster_array[rows[in_bounds], cols[in_bounds]]
    return values


towers_df["predicted_lst_c"] = sample_raster_at_points(
    baseline_lst, baseline_transform, towers_df["lon"].values, towers_df["lat"].values
)

n_outside = towers_df["predicted_lst_c"].isna().sum()
print(f"{n_outside} of {len(towers_df)} sites fall outside the raster's cropped extent "
      f"(dropped from risk analysis below)")

towers_df = towers_df.dropna(subset=["predicted_lst_c"]).reset_index(drop=True)
print(towers_df[["lon", "lat", "predicted_lst_c"]].describe())

## 4. Classify thermal risk tier

Thresholds below are a **design choice you should be able to justify/cite**, not a universal
standard -- outdoor telecom cabinets are commonly speced (ETSI/Telcordia-style ranges) to
operate reliably up to roughly 40-45 C ambient, with performance/lifetime derating above that,
and indoor exchange equipment is typically speced tighter. Adjust these to match whatever
equipment spec you're citing in your report.

- **Low risk:** predicted LST < 32 C
- **Medium risk:** 32-38 C
- **High risk:** > 38 C

In [ ]:
LOW_RISK_MAX = 32.0
MEDIUM_RISK_MAX = 38.0

def classify_risk(lst_c):
    if lst_c < LOW_RISK_MAX:
        return "Low"
    elif lst_c < MEDIUM_RISK_MAX:
        return "Medium"
    else:
        return "High"

towers_df["risk_tier"] = towers_df["predicted_lst_c"].apply(classify_risk)

print(towers_df["risk_tier"].value_counts())
print("\nShare of sites at High risk: {:.1f}%".format(
    (towers_df["risk_tier"] == "High").mean() * 100
))

## 5. (Optional) Adjust risk under a development scenario
Uses the Phase 4 scenario delta to show how many sites would shift into a higher risk tier if the simulated land-use change (e.g. the industrial park scenario) happened -- useful for framing this as a forward-looking resilience-planning tool, not just a snapshot of current conditions.

In [ ]:
if os.path.exists(SCENARIO_DELTA_PATH):
    with rasterio.open(SCENARIO_DELTA_PATH) as src:
        delta_lst = src.read(1)
        delta_transform = src.transform

    towers_df["scenario_delta_c"] = sample_raster_at_points(
        delta_lst, delta_transform, towers_df["lon"].values, towers_df["lat"].values
    )
    towers_df["scenario_delta_c"] = towers_df["scenario_delta_c"].fillna(0.0)
    towers_df["scenario_lst_c"] = towers_df["predicted_lst_c"] + towers_df["scenario_delta_c"]
    towers_df["scenario_risk_tier"] = towers_df["scenario_lst_c"].apply(classify_risk)

    tier_order = ["Low", "Medium", "High"]
    upgraded = (
        towers_df["risk_tier"].map(tier_order.index) <
        towers_df["scenario_risk_tier"].map(tier_order.index)
    )
    print(f"{upgraded.sum()} of {len(towers_df)} sites would shift to a higher risk tier "
          f"under this scenario")
    print(towers_df.groupby(["risk_tier", "scenario_risk_tier"]).size())
else:
    print(f"No scenario delta raster found at {SCENARIO_DELTA_PATH} -- skipping this step.")

## 6. Export results and a quick summary map

In [ ]:
out_csv = f"{telecom_dir}/telecom_heat_risk.csv"
towers_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(
    baseline_lst, cmap="inferno",
    extent=[baseline_bounds.left, baseline_bounds.right, baseline_bounds.bottom, baseline_bounds.top],
    origin="upper",
)
plt.colorbar(im, ax=ax, label="Predicted LST (C)", fraction=0.046)

risk_colors = {"Low": "deepskyblue", "Medium": "gold", "High": "red"}
for tier, color in risk_colors.items():
    subset = towers_df[towers_df["risk_tier"] == tier]
    ax.scatter(subset["lon"], subset["lat"], c=color, s=15, label=f"{tier} risk", edgecolors="black", linewidths=0.3)

ax.legend(loc="upper right")
ax.set_title("Telecom site thermal risk vs. predicted LST")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(f"{telecom_dir}/telecom_heat_risk_map.png", dpi=200)
plt.show()

print("Saved map:", f"{telecom_dir}/telecom_heat_risk_map.png")

## 7. Network coverage gap + composite site-suitability score

This adds the second telecom-specific factor: distance from every raster cell to the nearest
existing tower, as a proxy for network coverage gaps. Combined with thermal suitability (the
inverse of predicted LST -- cooler is more suitable), this produces a composite score that
answers a genuinely useful planning question: **where is it both thermally favorable and
under-served** -- i.e. good candidate territory for new infrastructure, as opposed to just
"where is it hot."

Distances are computed in approximate meters (equirectangular projection centered on the AOI)
rather than raw degrees, since 1 degree of longitude and 1 degree of latitude are not the same
physical distance -- using raw degree differences would distort the coverage-gap layer.

In [ ]:
from scipy.spatial import cKDTree

# Equirectangular approx: good enough for relative distance ranking at this scale
lat0 = np.deg2rad((baseline_bounds.top + baseline_bounds.bottom) / 2)
M_PER_DEG_LAT = 110540
M_PER_DEG_LON = 111320 * np.cos(lat0)

def lonlat_to_xy(lon, lat):
    x = (lon - baseline_bounds.left) * M_PER_DEG_LON
    y = (lat - baseline_bounds.bottom) * M_PER_DEG_LAT
    return x, y

tower_x, tower_y = lonlat_to_xy(towers_df["lon"].values, towers_df["lat"].values)
tower_tree = cKDTree(np.column_stack([tower_x, tower_y]))

# Build lon/lat for every raster pixel center
H, W = baseline_lst.shape
rows = np.arange(H)
cols = np.arange(W)
pixel_lons, pixel_lats = rasterio.transform.xy(baseline_transform, *np.meshgrid(rows, cols, indexing="ij"))
pixel_lons = np.asarray(pixel_lons)
pixel_lats = np.asarray(pixel_lats)

px_x, px_y = lonlat_to_xy(pixel_lons.ravel(), pixel_lats.ravel())
dist_m, _ = tower_tree.query(np.column_stack([px_x, px_y]))
coverage_gap_m = dist_m.reshape(H, W)

print("Coverage gap (distance to nearest tower) range: "
      f"{coverage_gap_m.min():.0f} m to {coverage_gap_m.max():.0f} m")

In [ ]:
def normalize(arr):
    valid = np.isfinite(arr)
    lo, hi = np.nanmin(arr[valid]), np.nanmax(arr[valid])
    return np.clip((arr - lo) / (hi - lo), 0, 1)

thermal_suitability = 1 - normalize(baseline_lst)   # cooler = more suitable
coverage_gap_norm = normalize(coverage_gap_m)        # farther from existing tower = more need

# Weights are a planning design choice -- justify/cite in your report, same as UHVI weights.
W_THERMAL = 0.5
W_COVERAGE = 0.5

suitability_score = W_THERMAL * thermal_suitability + W_COVERAGE * coverage_gap_norm

print("Composite suitability score range:", np.nanmin(suitability_score), "to", np.nanmax(suitability_score))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(baseline_lst, cmap="inferno")
axes[0].set_title("Predicted LST (C)")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(coverage_gap_m / 1000, cmap="viridis")
axes[1].set_title("Distance to nearest existing tower (km)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(suitability_score, cmap="YlGn")
axes[2].set_title("Composite site-suitability score\n(thermal + coverage gap)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig(f"{telecom_dir}/site_suitability_composite.png", dpi=200)
plt.show()

# Save the composite score as a georeferenced GeoTIFF, same pattern as Phase 5's exports
with rasterio.open(
    f"{telecom_dir}/site_suitability_composite.tif", "w", driver="GTiff",
    height=H, width=W, count=1, dtype="float32",
    crs=baseline_crs, transform=baseline_transform, nodata=np.nan, compress="lzw",
) as dst:
    dst.write(suitability_score.astype("float32"), 1)

print("Saved composite suitability raster for use in QGIS.")